# ATML PA2: LLM Post-Training - Kaggle Runner

This notebook provides a clean, modular execution runner for **ATML PA2**.

> **Fail-Safe Checkpointing:**
> After **EVERY** task, the notebook automatically packages and saves an incremental `.zip` file in `/kaggle/working/` (`task1_results.zip`, `task2_results.zip`, etc.).
> This ensures that **no progress is ever lost** even if a session disconnects, times out, or stops early.

---
### Recommended Kaggle Settings:
1. **Accelerator:** GPU T4 x 2 (or GPU P100)
2. **Internet:** **ON** (Required for Hugging Face and course assets)

## 0. Setup Environment & Clone Repository

In [1]:
# 1. Check GPU
!nvidia-smi

# 2. Clone latest code from GitHub
import os
if not os.path.exists("/kaggle/working/ATML-PA2"):
    !git clone https://github.com/moizejaz-31/ATML-PA2.git /kaggle/working/ATML-PA2
else:
    %cd /kaggle/working/ATML-PA2
    !git pull origin main

%cd /kaggle/working/ATML-PA2

Thu Oct  1 18:36:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 3. Install required pinned packages
!pip install -r requirements.txt -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.9/530.9 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.3 MB/s eta 0:00:00


In [3]:
# 4. Download and validate course assets (~1.05 GB, takes ~20 seconds on Kaggle)
!python -m scripts.download_assets
!python -m scripts.validate_assets

Source: AbDu11aHHH/ATML-PA2-assets
Revision: 0b350481fb03f5525a35bcdec4131bd4fe487f98
added_tokens.json: 100%|████████████████████████| 605/605 [00:00<00:00, 666kB/s]




adapter_config.json:   0%|                            | 0.00/860 [00:00<?, ?B/s]

grpo_k_cache.jsonl: 0.00B [00:00, ?B/s]


grpo_midpoint_metadata.json:   0%|                    | 0.00/308 [00:00<?, ?B/s]
adapter_config.json: 100%|██████████████████████| 860/860 [00:00<00:00, 316kB/s]
README.md: 5.22kB [00:00, 3.38MB/s]
grpo_k_cache.jsonl: 360kB [00:00, 57.3MB/s]
Fetching 90 files:   1%|▎                        | 1/90 [00:00<00:28,  3.17it/s]
checkpoints/grpo_midpoint_policy/adapter(…):   0%|  | 0.00/4.37M [00:00<?, ?B/s]

cached/ppo_rollout.pt:   0%|                         | 0.00/176k [00:00<?, ?B/s]


cached/ppo_rollout_salvage_raw.pt:   0%|             | 0.00/329k [00:00<?, ?B/s]



chat_template.jinja: 2.51kB [00:00, 12.1MB/s]




special_tokens_map.json: 100%|█████████████████| 613/613 [00:00<00:00, 4.53MB/s]




--- 
## Task 1: Direct Preference Optimization (DPO)
1. **Standard DPO:** Train 1 epoch on UltraFeedback preference data
2. **Evaluate Standard DPO:** Compute held-out preference accuracy, KL drift, RM score, length
3. **Regularization Ablation:** Sweep $\beta \in \{0.03, 0.10, 0.30\}$
4. **Length Confounding:** Train on length-balanced data and test word-limit compliance

In [4]:
print("=== Task 1.1: Standard DPO Training ===")
!python -m task1_dpo.train --config configs/dpo.yaml --run-name standard

print("\n=== Task 1.2: Evaluate Standard DPO ===")
!python -m task1_dpo.evaluate --config configs/dpo.yaml --adapter outputs/task1_dpo/standard --name standard

print("\n=== Task 1.3: Beta Regularization Ablation ===")
!python -m task1_dpo.ablate_beta --config configs/dpo.yaml

print("\n=== Task 1.4: Length Confounding & Word-Limit Study ===")
!python -m task1_dpo.analyze_length --config configs/dpo.yaml

# SAFEGUARD: Immediately package and save Task 1 results so they are permanently preserved
!zip -r /kaggle/working/task1_results.zip results/task1_dpo/ report/figures/ outputs/task1_dpo/
print("\n[SAFEGUARD] Successfully saved /kaggle/working/task1_results.zip!")

=== Task 1.1: Standard DPO Training ===
tokenizer_config.json: 7.30kB [00:00, 24.9MB/s]
vocab.json: 2.78MB [00:00, 40.1MB/s]
merges.txt: 1.67MB [00:00, 124MB/s]
tokenizer.json: 7.03MB [00:00, 168MB/s]
config.json: 100%|█████████████████████████████| 660/660 [00:00<00:00, 3.76MB/s]
model.safetensors: 100%|████████████████████| 3.09G/3.09G [00:08<00:00, 366MB/s]
generation_config.json: 100%|██████████████████| 242/242 [00:00<00:00, 1.62MB/s]
[DPO train] run=standard beta=0.1 epochs=1 grad_accum=8 trainable=1,089,536/1,544,803,840
[DPO train] dataset=1500 examples
[DPO train] output -> /kaggle/working/ATML-PA2/outputs/task1_dpo/standard
[DPO train] log    -> /kaggle/working/ATML-PA2/results/task1_dpo/dpo_train_standard.jsonl
  [step    1] loss=0.6931 pref_acc=0.000 logit_mean=0.000 kl=0.0000 grad_norm=2.040 time=16.5s
           peak_vram=7276MB
  [step   10] loss=0.6786 pref_acc=1.000 logit_mean=0.029 kl=0.0057 grad_norm=2.487 time=148.9s
           peak_vram=7283MB
  [step   20] loss=0.

--- 
## Task 2: Proximal Policy Optimization (PPO)
Continues from course-supplied midpoint checkpoints (`checkpoints/ppo_midpoint_policy`, `checkpoints/ppo_midpoint_value`).

In [5]:
print("=== Task 2.1: Standard PPO Continuation (20 updates) ===")
!python -m task2_ppo.continue_train --config configs/ppo.yaml --run-name standard

print("\n=== Task 2.2: Evaluate PPO Policy ===")
!python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter outputs/task2_ppo/standard --name standard

print("\n=== Task 2.3: Clipping Study (epsilon in {0.05, 0.20, 0.50}) ===")
!python -m task2_ppo.analyze_clipping --config configs/ppo.yaml

print("\n=== Task 2.4: KL-Pressure Study (beta_KL in {0, 0.10, 0.20}) ===")
!python -m task2_ppo.ablate_kl --config configs/ppo.yaml

# SAFEGUARD: Immediately package and save Task 2 results
!zip -r /kaggle/working/task2_results.zip results/task2_ppo/ report/figures/ outputs/task2_ppo/
print("\n[SAFEGUARD] Successfully saved /kaggle/working/task2_results.zip!")

=== Task 2.1: Standard PPO Continuation (20 updates) ===
[PPO Continuation] run=standard updates=20 eps=0.2 beta_kl=0.1
Policy trainable params: 1,089,536
Value trainable params:  541,568
Output adapter:          /kaggle/working/ATML-PA2/outputs/task2_ppo/standard
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/ATML-PA2/task2_ppo/continue_train.py", line 363, in <module>
    main()
  File "/kaggle/working/ATML-PA2/task2_ppo/continue_train.py", line 359, in main
    run_ppo(args.config, args.output, args.updates, args.clip_epsilon, args.kl_beta, args.run_name)
  File "/kaggle/working/ATML-PA2/task2_ppo/continue_train.py", line 267, in run_ppo
    new_logp, _ = response_token_logprobs(policy, sequences, attention_mask, prompt_width, response_ids)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/

--- 
## Task 3: Group Relative Policy Optimization (GRPO)
Critic-free RL continuing from `checkpoints/grpo_midpoint_policy`.

In [6]:
print("=== Task 3.1: Standard GRPO Continuation (20 updates) ===")
!python -m task3_grpo.continue_train --config configs/grpo.yaml --run-name standard

print("\n=== Task 3.2: Evaluate GRPO Policy ===")
!python -m task3_grpo.evaluate --config configs/grpo.yaml --adapter outputs/task3_grpo/standard --name standard

print("\n=== Task 3.3: Group Size Study (K in {2, 4, 8}) ===")
!python -m task3_grpo.analyze_group_size --config configs/grpo.yaml

print("\n=== Task 3.4: Normalization Study (Canonical vs Dr. GRPO) ===")
!python -m task3_grpo.compare_normalization --config configs/grpo.yaml

# SAFEGUARD: Immediately package and save Task 3 results
!zip -r /kaggle/working/task3_results.zip results/task3_grpo/ report/figures/ outputs/task3_grpo/
print("\n[SAFEGUARD] Successfully saved /kaggle/working/task3_results.zip!")

=== Task 3.1: Standard GRPO Continuation (20 updates) ===
[GRPO Continuation] run=standard updates=20 K=4 loss_type=grpo
Policy trainable params: 1,089,536
Output adapter:          /kaggle/working/ATML-PA2/outputs/task3_grpo/standard
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/ATML-PA2/task3_grpo/continue_train.py", line 314, in <module>
    main()
  File "/kaggle/working/ATML-PA2/task3_grpo/continue_train.py", line 310, in main
    run_grpo(args.config, args.output, args.updates, args.loss_type, args.run_name)
  File "/kaggle/working/ATML-PA2/task3_grpo/continue_train.py", line 222, in run_grpo
    new_logp, _ = response_token_logprobs(policy, sequences, attention_mask, prompt_width, response_ids)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/ATML-PA2/common/generation.py", line 97, in

--- 
## Task 5: Feedback Source (RLVR vs RLAIF)
Evaluates frozen course checkpoints (`checkpoints/rlvr_policy` and `checkpoints/rlaif_policy`). Independent of Tasks 1-3.

In [7]:
print("=== Task 5.1: In-Domain GSM8K Math Evaluation ===")
!python -m task5_feedback.evaluate_math --config configs/feedback.yaml --dataset gsm

print("\n=== Task 5.2: Controlled Reward Diagnostics (100-pair perturbations) ===")
!python -m task5_feedback.score_perturbations --config configs/feedback.yaml

print("\n=== Task 5.3: Out-of-Domain SVAMP Transfer Evaluation ===")
!python -m task5_feedback.evaluate_math --config configs/feedback.yaml --dataset transfer

print("\n=== Task 5.4: Compare Feedback Mechanisms ===")
!python -m task5_feedback.compare_feedback --config configs/feedback.yaml

# SAFEGUARD: Immediately package and save Task 5 results
!zip -r /kaggle/working/task5_results.zip results/task5_feedback/ report/figures/
print("\n[SAFEGUARD] Successfully saved /kaggle/working/task5_results.zip!")

=== Task 5.1: In-Domain GSM8K Math Evaluation ===
Task 5: Math Reasoning Evaluation (GSM)
Evaluating 300 test problems across: ['sft', 'rlvr', 'rlaif']

Generating responses for policy: SFT...
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  SFT: Exact Acc = 0.0%, Format = 0.0%, Mean Length = 287.8 tokens

Generating responses for policy: RLVR...
  RLVR: Exact Acc = 0.0%, Format = 0.0%, Mean Length = 288.6 tokens

Generating responses for policy: RLAIF...
  RLAIF: Exact Acc = 0.0%, Format = 0.0%, Mean Length = 288.6 tokens
tokenizer_config.json: 7.30kB [00:00, 16.7MB/s]
vocab.json: 2.78MB [00:00, 44.3MB/s]
merges.txt: 1.67MB [00:00, 129MB/s]
tokenizer.json: 7.03MB [00:00, 168MB/s]
config.json: 100%|█████████████████████████████| 661/661 [00:00<00:00, 3.06MB/s]
model.safetensors.index.json: 35.6kB [00:00, 99.0MB/s]
Fetching 2 files:   0%|                                   | 0/2 [00:0

--- 
## Task 4: Safety Calibration
Compares SFT baseline, standard DPO, standard PPO, and standard GRPO policies on XSTest prompts.

In [8]:
print("=== Task 4.1: Generate Responses for 4 Policies on XSTest ===")
!python -m task4_safety.generate_responses --config configs/feedback.yaml

print("\n=== Task 4.2: Judge Responses with AI Safety Judge ===")
!python -m task4_safety.judge_responses --config configs/feedback.yaml

print("\n=== Task 4.3: Generate 60-example Manual Audit Sheet ===")
!python -m task4_safety.make_audit_sheet --config configs/feedback.yaml

print("\n=== Task 4.4: Safety Calibration Metrics & Disagreements ===")
!python -m task4_safety.evaluate_safety --config configs/feedback.yaml

# SAFEGUARD: Immediately package and save Task 4 results
!zip -r /kaggle/working/task4_results.zip results/task4_safety/ report/figures/
print("\n[SAFEGUARD] Successfully saved /kaggle/working/task4_results.zip!")

=== Task 4.1: Generate Responses for 4 Policies on XSTest ===
Task 4: Generating Deterministic Responses on XSTest
Policies to evaluate: ['sft', 'dpo', 'ppo', 'grpo']
Output directory:     /kaggle/working/ATML-PA2/results/task4_safety

Processing policy 'sft' -> /kaggle/working/ATML-PA2/results/task4_safety/generated_sft.jsonl...
Generating deterministic responses for policy 'sft' (450 prompts)...
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Wrote 450 responses to /kaggle/working/ATML-PA2/results/task4_safety/generated_sft.jsonl

Processing policy 'dpo' -> /kaggle/working/ATML-PA2/results/task4_safety/generated_dpo.jsonl...
Generating deterministic responses for policy 'dpo' (450 prompts)...
Wrote 450 responses to /kaggle/working/ATML-PA2/results/task4_safety/generated_dpo.jsonl

Processing policy 'ppo' -> /kaggle/working/ATML-PA2/results/task4_safety/generated_ppo.jsonl...
Traceb

--- 
## Master Aggregation & Complete Results Packaging

In [9]:
# Run master visualization to compile all figures and summary tables
!python -m common.visualize_all

# Package everything across all tasks into one master archive
!zip -r /kaggle/working/ATML_PA2_ALL_RESULTS.zip results/ report/figures/ outputs/
print("\n🎉 ALL DONE! You can now download individual task zips or 'ATML_PA2_ALL_RESULTS.zip' from the Output tab.")

ATML PA2 — COMPLETE POST-TRAINING RESEARCH DASHBOARD
Repo Root:    /kaggle/working/ATML-PA2
Results Dir:  /kaggle/working/ATML-PA2/results
Figures Dir:  /kaggle/working/ATML-PA2/report/figures

TASK 1: DIRECT PREFERENCE OPTIMIZATION (DPO) SUMMARY
Standard DPO Run: 94 steps, final loss = 0.6530, final pref acc = 100.0%

[RQ1: Regularization Strength β]
     β |   Held-out Acc |  Policy Drift (KL) |    Mean Reward |   Length (words)
--------------------------------------------------------------------------
  0.03 |         60.0% |           -0.0718 |         1.071 |           112.6
  0.10 |         61.7% |           -0.0591 |         1.087 |           111.7
  0.30 |         63.3% |           -0.0571 |         1.127 |           110.2

[RQ2: Length Confounding & Bias]
Stratum              |   Standard DPO Acc |    Length-Balanced Acc
-----------------------------------------------------------------
preferred_longer     |             47.6% |                 42.7%
matched_length       |     